# Youtube Downloading
YT-DLP: https://github.com/yt-dlp/yt-dlp  
FFmpeg: https://www.ffmpeg.org/download.html


In [1]:
import os
import subprocess
import re

In [2]:
!python --version
!where python
!pip --version
# !where pip

Python 3.12.10
c:\Python\Python312\python.exe
C:\Python\Python313\python.exe
C:\Program Files\Inkscape\bin\python.exe
C:\Users\Ricardo\AppData\Local\Microsoft\WindowsApps\python.exe
pip 25.3 from C:\Python\Python312\Lib\site-packages\pip (python 3.12)



In [3]:
# Check current directory and list contents
import os
print("Current directory:", os.getcwd())
print("Contents:", os.listdir())

Current directory: d:\Utility\media
Contents: ['.git', '.gitignore', 'audio', 'ffmpeg-8.0.1-essentials_build', 'loop_n_fading.ipynb', 'Past Lives_1hour.mp3', 'README.md', 'twitter_cookies.txt', 'unlisted', 'youtube_cookies.txt', 'yt-dlp', 'yt-dlp.ipynb', 'yt-dlp_s4.ipynb']


In [4]:
# Method 1: Using shell command with full path
# Linux/MacOS
# !./yt-dlp/yt-dlp.exe --version
# Windows
!yt-dlp\yt-dlp.exe --version  # Use backslash for Windows paths
print()
!ffmpeg-8.0.1-essentials_build\bin\ffmpeg.exe -version

# Method 2: For playlist URLs, use this format:
# !./yt-dlp/yt-dlp.exe --flat-playlist --print "%(url)s" "https://www.youtube.com/playlist?list=YOUR_PLAYLIST_ID"

# Example for downloading:
# !./yt-dlp/yt-dlp.exe -f "best[height<=720]" -o "downloads/%(title)s.%(ext)s" "YOUR_URL_HERE"

2026.03.03

ffmpeg version 8.0.1-essentials_build-www.gyan.dev Copyright (c) 2000-2025 the FFmpeg developers
built with gcc 15.2.0 (Rev8, Built by MSYS2 project)
configuration: --enable-gpl --enable-version3 --enable-static --disable-w32threads --disable-autodetect --enable-fontconfig --enable-iconv --enable-gnutls --enable-libxml2 --enable-gmp --enable-bzlib --enable-lzma --enable-zlib --enable-libsrt --enable-libssh --enable-libzmq --enable-avisynth --enable-sdl2 --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxvid --enable-libaom --enable-libopenjpeg --enable-libvpx --enable-mediafoundation --enable-libass --enable-libfreetype --enable-libfribidi --enable-libharfbuzz --enable-libvidstab --enable-libvmaf --enable-libzimg --enable-amf --enable-cuda-llvm --enable-cuvid --enable-dxva2 --enable-d3d11va --enable-d3d12va --enable-ffnvcodec --enable-libvpl --enable-nvdec --enable-nvenc --enable-vaapi --enable-openal --enable-libgme --enable-libopenmpt --enable-libopencore-amr

In [5]:
# WORKING ROUTE FOR YT-DLP
# =========================

# Step 1: Define the yt-dlp executable path
# Set the path to yt-dlp executable (we're already in the yt-dlp directory)
ytdlp_path = "yt-dlp\yt-dlp.exe"
ffmpeg_path = "ffmpeg-8.0.1-essentials_build\\bin"  # "ffmpeg-7.1.1-essentials_build\\bin" 

# Step 2: Create helper function for running yt-dlp commands
def run_ytdlp(args):
    """Helper function to run yt-dlp commands"""
    cmd = [ytdlp_path] + args + ["--ffmpeg-location", ffmpeg_path]
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, cwd=os.getcwd())
        print("STDOUT:", result.stdout)
        if result.stderr:
            print("STDERR:", result.stderr)
        return result
    except Exception as e:
        print(f"Error running yt-dlp: {e}")
        return None

# Step 3: Test the setup
print("Testing yt-dlp setup...")
run_ytdlp(["--version"])

Testing yt-dlp setup...


<>:6: SyntaxWarning: invalid escape sequence '\y'
<>:6: SyntaxWarning: invalid escape sequence '\y'
C:\Users\Ricardo\AppData\Local\Temp\ipykernel_13468\1210210038.py:6: SyntaxWarning: invalid escape sequence '\y'
  ytdlp_path = "yt-dlp\yt-dlp.exe"


STDOUT: 2026.03.03



CompletedProcess(args=['yt-dlp\\yt-dlp.exe', '--version', '--ffmpeg-location', 'ffmpeg-8.0.1-essentials_build\\bin'], returncode=0, stdout='2026.03.03\n', stderr='')

In [6]:
def list_playlist_urls(completed_process):
    """List all URLs in a YouTube playlist"""
    if completed_process and completed_process.stdout:
        urls = completed_process.stdout.strip().split('\n')
        return urls
    else:
        print("No URLs found or error occurred.")
        return []

```python
# Chỉ title (hiện tại)
"%(title)s.%(ext)s"

# Title + ID (như file cũ)
"%(title)s [%(id)s].%(ext)s"

# Title + Uploader + ID
"%(uploader)s - %(title)s [%(id)s].%(ext)s"

# Date + Title + ID
"%(upload_date)s - %(title)s [%(id)s].%(ext)s"
```

In [7]:
# PRACTICAL EXAMPLES
# ==================

# Example 1: Get video info
def get_video_info(url):
    """Get video information"""
    args = ["--dump-json", url]
    return run_ytdlp(args)

# Example 2: List playlist URLs
def get_playlist_urls(playlist_url):
    """List all URLs in a playlist"""
    args = ["--flat-playlist", "--print", "%(url)s", playlist_url]
    return run_ytdlp(args)

# Example 3: Download with specific format
def download_video(url, output_dir="downloads"):
    """Download video with specific settings"""
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    args = [
        "-f", "best[height<=1080]",  # Max 1080p quality
        "-o", f"D:/downloaded_media/{output_dir}/%(title)s [%(id)s].%(ext)s",  # Output filename pattern
        "--restrict-filenames", # ✅ GIẢI PHÁP: Bỏ dấu tiếng Việt và ký tự đặc biệt trong tên file
        url
    ]
    return run_ytdlp(args)

# Example 4: Download audio only
def download_audio(url, output_dir="audio"):
    """Download audio only"""
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    args = [
        "-x",  # Extract audio
        "--audio-format", "best",  # Convert to format in list [best (default), aac, alac, flac, m4a, mp3, opus, vorbis, wav]
        "-o", f"D:/downloaded_media/{output_dir}/%(title)s [%(id)s].%(ext)s",
        "--restrict-filenames", # ✅ GIẢI PHÁP: Bỏ dấu tiếng Việt và ký tự đặc biệt trong tên file
        url
    ]
    return run_ytdlp(args)

# Example 5: Download best video (Video + Audio)
def download_best_video2(url, output_dir="unlisted"):
    """Download best video (Video + Audio)"""
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    args = [
        "-f", "bestvideo+bestaudio/best",  # Best video + best audio
        "-o", f"D:/downloaded_media/{output_dir}/%(title)s [%(id)s].%(ext)s",
        "--restrict-filenames", # ✅ GIẢI PHÁP: Bỏ dấu tiếng Việt và ký tự đặc biệt trong tên file
        url
    ]
    return run_ytdlp(args)

def download_best_video(
    url,
    output_dir="unlisted",
    cookies_file="twitter_cookies.txt"
):
    """Download best video (Video + Audio) from X/Twitter (Brave-safe)"""

    # Normalize domain
    url = re.sub(r"https?://(www\.)?x\.com", "https://twitter.com", url)

    base_dir = f"D:/downloaded_media/{output_dir}"
    os.makedirs(base_dir, exist_ok=True)

    args = [
        "--cookies", cookies_file,              # ✅ ONLY THIS
        "--no-playlist",
        "--merge-output-format", "mp4",
        # f-string cho output để tránh lỗi đường dẫn Windows
        "-o", f"{base_dir}/%(title)s [%(id)s].%(ext)s",

        # ✅ GIẢI PHÁP: Bỏ dấu tiếng Việt và ký tự đặc biệt trong tên file
        "--restrict-filenames", 
        
        # ✅ Đảm bảo encoding là UTF-8 để nhận diện tiêu đề đúng trước khi format
        "--recode-video", "mp4",
        
        "-f", "bestvideo+bestaudio/best",
        url
    ]

    return run_ytdlp(args)


print("Helper functions defined successfully!")
print("Available functions:")
print("- get_video_info(url)")
print("- get_playlist_urls(playlist_url)")
print("- download_video(url, output_dir='downloads')")
print("- download_audio(url, output_dir='audio')")
print("- download_unlisted_video(url, output_dir='unlisted')")

Helper functions defined successfully!
Available functions:
- get_video_info(url)
- get_playlist_urls(playlist_url)
- download_video(url, output_dir='downloads')
- download_audio(url, output_dir='audio')
- download_unlisted_video(url, output_dir='unlisted')


In [8]:
# USAGE EXAMPLES
# ===============

# Example: Test with a playlist (replace with your actual playlist URL)
# playlist_url = "https://www.youtube.com/playlist?list=YOUR_PLAYLIST_ID"
# list_playlist_urls(playlist_url)

# Example: Test with a single video (replace with your actual video URL)
# video_url = "https://www.youtube.com/watch?v=YOUR_VIDEO_ID"
# get_video_info(video_url)

# Example: Download a video
# download_video(video_url, "downloads")

# Example: Download audio only
# download_audio(video_url, "audio")

# For your specific use case, to list playlist URLs:
print("To use with a playlist, replace YOUR_PLAYLIST_ID and run:")
print('list_playlist_urls("https://www.youtube.com/playlist?list=YOUR_PLAYLIST_ID")')

print("\nTo download videos from your existing folders:")
print("Current audio folders found:")
for folder in ["edm", "edm - no lyric", "remix"]:
    folder = os.path.join("audio", folder)
    if os.path.exists(folder):
        files = [f for f in os.listdir(folder) if f.endswith('.opus')]
        print(f"  {folder}: {len(files)} files")
        if files:
            print(f"    Example: {files[0]}")

To use with a playlist, replace YOUR_PLAYLIST_ID and run:
list_playlist_urls("https://www.youtube.com/playlist?list=YOUR_PLAYLIST_ID")

To download videos from your existing folders:
Current audio folders found:


### Download one audio

In [9]:
url = input("Enter Youtube URL to download audio: ")
download_audio(url)

STDOUT: [youtube] Extracting URL: https://youtu.be/2B9jv7prFno?si=VekCYNu7hpLbRNC6
[youtube] 2B9jv7prFno: Downloading webpage
[youtube] 2B9jv7prFno: Downloading android vr player API JSON
[info] 2B9jv7prFno: Downloading 1 format(s): 251
[download] Destination: D:\downloaded_media\audio\Memory_of_Lightwaves_~_Relaxing_FFX-2_Music_w_Ocean_and_Soft_Rain_Sounds [2B9jv7prFno].webm

[download]   0.0% of   31.50MiB at  867.85KiB/s ETA 00:37
[download]   0.0% of   31.50MiB at    2.54MiB/s ETA 00:12
[download]   0.0% of   31.50MiB at    3.02MiB/s ETA 00:10
[download]   0.0% of   31.50MiB at    6.47MiB/s ETA 00:04
[download]   0.1% of   31.50MiB at  189.31KiB/s ETA 02:50
[download]   0.2% of   31.50MiB at  381.86KiB/s ETA 01:24
[download]   0.4% of   31.50MiB at  609.27KiB/s ETA 00:52
[download]   0.8% of   31.50MiB at    1.05MiB/s ETA 00:29
[download]   1.6% of   31.50MiB at    1.68MiB/s ETA 00:18
[download]   3.2% of   31.50MiB at    1.73MiB/s ETA 00:17
[download]   6.3% of   31.50MiB at    1.

CompletedProcess(args=['yt-dlp\\yt-dlp.exe', '-x', '--audio-format', 'best', '-o', 'D:/downloaded_media/audio/%(title)s [%(id)s].%(ext)s', '--restrict-filenames', 'https://youtu.be/2B9jv7prFno?si=VekCYNu7hpLbRNC6', '--ffmpeg-location', 'ffmpeg-8.0.1-essentials_build\\bin'], returncode=0, stdout='[youtube] Extracting URL: https://youtu.be/2B9jv7prFno?si=VekCYNu7hpLbRNC6\n[youtube] 2B9jv7prFno: Downloading webpage\n[youtube] 2B9jv7prFno: Downloading android vr player API JSON\n[info] 2B9jv7prFno: Downloading 1 format(s): 251\n[download] Destination: D:\\downloaded_media\\audio\\Memory_of_Lightwaves_~_Relaxing_FFX-2_Music_w_Ocean_and_Soft_Rain_Sounds [2B9jv7prFno].webm\n\n[download]   0.0% of   31.50MiB at  867.85KiB/s ETA 00:37\n[download]   0.0% of   31.50MiB at    2.54MiB/s ETA 00:12\n[download]   0.0% of   31.50MiB at    3.02MiB/s ETA 00:10\n[download]   0.0% of   31.50MiB at    6.47MiB/s ETA 00:04\n[download]   0.1% of   31.50MiB at  189.31KiB/s ETA 02:50\n[download]   0.2% of   31.5

### Download a list of audio

In [10]:
url = input("Enter Youtube playlist URL to download audios: ")
completed_process = get_playlist_urls(url)
lst_urls = list_playlist_urls(completed_process)
print("Playlist URLs:", lst_urls)

STDOUT: 
STDERR: ERROR: [generic] '' is not a valid URL

No URLs found or error occurred.
Playlist URLs: []


In [11]:
for url in lst_urls:
    print(f"Downloading audio for: {url}")
    download_audio(url)

### Download a/an (unlisted) video

In [8]:
url = input("Enter best video URL to download: ")
download_best_video(url, output_dir='unlisted')

STDOUT: [youtube] Extracting URL: https://www.youtube.com/watch?v=bLCkrn2BeF8
[youtube] bLCkrn2BeF8: Downloading webpage
[youtube] bLCkrn2BeF8: Downloading android vr player API JSON
[info] bLCkrn2BeF8: Downloading 1 format(s): 137+251
[download] Destination: D:\downloaded_media\unlisted\TO_L_2_-_H_ng_d_n_lam_Th_c_tru_-_ARC_online [bLCkrn2BeF8].f137.mp4

[download]   0.0% of   46.65MiB at  498.31KiB/s ETA 01:35
[download]   0.0% of   46.65MiB at 1000.87KiB/s ETA 00:47
[download]   0.0% of   46.65MiB at    1.71MiB/s ETA 00:27
[download]   0.0% of   46.65MiB at    2.93MiB/s ETA 00:15
[download]   0.1% of   46.65MiB at    5.05MiB/s ETA 00:09
[download]   0.1% of   46.65MiB at    3.62MiB/s ETA 00:12
[download]   0.3% of   46.65MiB at    4.00MiB/s ETA 00:11
[download]   0.5% of   46.65MiB at    4.53MiB/s ETA 00:10
[download]   1.1% of   46.65MiB at    6.01MiB/s ETA 00:07
[download]   2.1% of   46.65MiB at    7.87MiB/s ETA 00:05
[download]   4.3% of   46.65MiB at    9.26MiB/s ETA 00:04
[down

CompletedProcess(args=['yt-dlp\\yt-dlp.exe', '--cookies', 'twitter_cookies.txt', '--no-playlist', '--merge-output-format', 'mp4', '-o', 'D:/downloaded_media/unlisted/%(title)s [%(id)s].%(ext)s', '--restrict-filenames', '--recode-video', 'mp4', '-f', 'bestvideo+bestaudio/best', 'https://www.youtube.com/watch?v=bLCkrn2BeF8', '--ffmpeg-location', 'ffmpeg-8.0.1-essentials_build\\bin'], returncode=0, stdout='[youtube] Extracting URL: https://www.youtube.com/watch?v=bLCkrn2BeF8\n[youtube] bLCkrn2BeF8: Downloading webpage\n[youtube] bLCkrn2BeF8: Downloading android vr player API JSON\n[info] bLCkrn2BeF8: Downloading 1 format(s): 137+251\n[download] Destination: D:\\downloaded_media\\unlisted\\TO_L_2_-_H_ng_d_n_lam_Th_c_tru_-_ARC_online [bLCkrn2BeF8].f137.mp4\n\n[download]   0.0% of   46.65MiB at  498.31KiB/s ETA 01:35\n[download]   0.0% of   46.65MiB at 1000.87KiB/s ETA 00:47\n[download]   0.0% of   46.65MiB at    1.71MiB/s ETA 00:27\n[download]   0.0% of   46.65MiB at    2.93MiB/s ETA 00:15\